# Floyd-Warshall: All-Pairs Matrix Simulator

Floyd-Warshall finds the cheapest path between every pair of places in a weighted graph.

The algorithm is a classic example of dynamic programming from the early 1960s, associated with Robert Floyd and Stephen Warshall. It is useful when the question is not one route, but a whole table of relationships: transit times, network latency, prerequisite reachability, or graph analysis.

In this notebook, you will build it with small objects: places, routes, a distance matrix, and a runner that updates the matrix using dynamic programming.

<details>
<summary>Big idea</summary>

Instead of asking for one shortest path, Floyd-Warshall asks: what if each place gets a turn as a possible middle stop?

</details>

## 1. The Mental Model

Floyd-Warshall is a dynamic programming algorithm:

- **Place**: a node, like `Arcade` or `Library`
- **Route**: a directed edge with a travel cost
- **Matrix**: a table of best known costs from every start to every destination
- **Via place**: the middle stop currently being tested
- **Update rule**: keep the cheaper of direct cost and `start -> via -> destination`

The algorithm repeats one move: let each place act as a possible middle stop, then improve the matrix if that stop creates a cheaper route.

<details>
<summary>Hint: the formula</summary>

For each start `i`, destination `j`, and middle stop `k`:

`distance[i][j] = min(distance[i][j], distance[i][k] + distance[k][j])`

</details>

## 2. Build the Objects

Implementation plan:

1. `Place` stores a location name.
2. `RouteMap` stores one-way route costs.
3. `MatrixStep` records each dynamic programming pass.
4. `FloydWarshallRunner` owns the matrix update logic.
5. `MatrixPrinter` keeps the matrix readable.

<details>
<summary>Implementation hint</summary>

Start the matrix with `0` on the diagonal, direct route costs where routes exist, and `inf` everywhere else.

</details>

**Object model.** Define `Place`, the named objects used by the next examples.


In [ ]:
from dataclasses import dataclass, field

from math import inf

@dataclass(frozen=True, order=True)
class Place:
    name: str

    def __str__(self) -> str:
        return self.name

    def __format__(self, spec: str) -> str:
        return format(self.name, spec)


**Object model.** Define `RouteMap`, the named objects used by the next examples.


In [ ]:
@dataclass
class RouteMap:
    routes: dict[Place, dict[Place, float]] = field(default_factory=dict)

    def add_place(self, place: Place) -> None:
        self.routes.setdefault(place, {})

    def connect(self, start: Place, end: Place, cost: float, two_way: bool = False) -> None:
        self.add_place(start)
        self.add_place(end)
        self.routes[start][end] = cost

        if two_way:
            self.routes[end][start] = cost

    def places(self) -> list[Place]:
        return sorted(self.routes)

    def direct_cost(self, start: Place, end: Place) -> float:
        if start == end:
            return 0
        return self.routes.get(start, {}).get(end, inf)

    def describe(self) -> str:
        rows = []
        for start in self.places():
            targets = self.routes[start]
            label = ", ".join(f"{end}({cost:g})" for end, cost in sorted(targets.items())) or "no outgoing routes"
            rows.append(f"{start:>8} -> {label}")
        return "\n".join(rows)


**Trace model.** Define `MatrixStep`, `FloydWarshallResult`, the structure used to capture replayable algorithm state.


In [ ]:
@dataclass
class MatrixStep:
    via: Place | None
    distances: list[list[float]]
    changes: list[tuple[Place, Place, float, float]]
    note: str

@dataclass
class FloydWarshallResult:
    places: list[Place]
    distances: list[list[float]]
    next_hop: dict[tuple[Place, Place], Place | None]
    steps: list[MatrixStep]


**Algorithm engine.** Define `FloydWarshallRunner`, the class that runs the main simulation or algorithm.


In [ ]:
class FloydWarshallRunner:
    def __init__(self, route_map: RouteMap):
        self.route_map = route_map

    def run(self) -> FloydWarshallResult:
        places = self.route_map.places()
        if not places:
            raise ValueError("Floyd-Warshall needs at least one place.")

        index = {place: position for position, place in enumerate(places)}
        distances = [[self.route_map.direct_cost(start, end) for end in places] for start in places]
        next_hop: dict[tuple[Place, Place], Place | None] = {}

        for start in places:
            for end in places:
                if start == end:
                    next_hop[(start, end)] = end
                elif self.route_map.direct_cost(start, end) < inf:
                    next_hop[(start, end)] = end
                else:
                    next_hop[(start, end)] = None

        steps = [self._snapshot(None, distances, [], "Start with direct route costs.")]

        for via in places:
            via_index = index[via]
            changes = []

            for start in places:
                start_index = index[start]
                for end in places:
                    end_index = index[end]
                    through_via = distances[start_index][via_index] + distances[via_index][end_index]

                    if through_via < distances[start_index][end_index]:
                        old_cost = distances[start_index][end_index]
                        distances[start_index][end_index] = through_via
                        next_hop[(start, end)] = next_hop[(start, via)]
                        changes.append((start, end, old_cost, through_via))

            steps.append(self._snapshot(via, distances, changes, f"Use {via} as an allowed middle stop."))

        self._raise_if_negative_cycle(places, distances)
        return FloydWarshallResult(places=places, distances=distances, next_hop=next_hop, steps=steps)

    def path_between(self, result: FloydWarshallResult, start: Place, end: Place) -> list[Place]:
        if result.next_hop[(start, end)] is None:
            return []

        path = [start]
        current = start
        while current != end:
            current = result.next_hop[(current, end)]
            if current is None:
                return []
            path.append(current)

        return path

    def _snapshot(
        self,
        via: Place | None,
        distances: list[list[float]],
        changes: list[tuple[Place, Place, float, float]],
        note: str,
    ) -> MatrixStep:
        return MatrixStep(
            via=via,
            distances=[row.copy() for row in distances],
            changes=changes.copy(),
            note=note,
        )

    def _raise_if_negative_cycle(self, places: list[Place], distances: list[list[float]]) -> None:
        for position, place in enumerate(places):
            if distances[position][position] < 0:
                raise ValueError(f"Negative cycle detected around {place}.")


**Object model.** Define `MatrixPrinter`, the named objects used by the next examples.


In [ ]:
class MatrixPrinter:
    def __init__(self, places: list[Place]):
        self.places = places

    def matrix(self, distances: list[list[float]]) -> str:
        header = " " * 10 + " ".join(f"{place.name[:7]:>7}" for place in self.places)
        rows = [header]

        for place, costs in zip(self.places, distances):
            cells = " ".join(f"{self._format_cost(cost):>7}" for cost in costs)
            rows.append(f"{place.name[:9]:>9} {cells}")

        return "\n".join(rows)

    def path(self, path: list[Place]) -> str:
        return " -> ".join(str(place) for place in path) or "no route"

    def changes(self, changes: list[tuple[Place, Place, float, float]]) -> str:
        if not changes:
            return "no matrix cells improved"

        return "; ".join(
            f"{start}->{end}: {self._format_cost(old)} to {self._format_cost(new)}"
            for start, end, old, new in changes
        )

    def _format_cost(self, cost: float) -> str:
        return "inf" if cost == inf else f"{cost:g}"


## 3. Create a Tiny Route Map

Now make a directed route map. A route from `Arcade` to `Bakery` does not automatically mean there is a route back.

<details>
<summary>Hint: where is the matrix?</summary>

Rows are starting places. Columns are destinations. Each cell stores the cheapest known cost from that row to that column.

</details>

In [2]:
arcade = Place("Arcade")
bakery = Place("Bakery")
cafe = Place("Cafe")
diner = Place("Diner")
library = Place("Library")

city = RouteMap()
city.connect(arcade, bakery, 3)
city.connect(arcade, cafe, 10)
city.connect(arcade, library, 20)
city.connect(bakery, cafe, 2)
city.connect(bakery, library, 9)
city.connect(cafe, diner, 1)
city.connect(diner, bakery, 1)
city.connect(diner, library, 2)
city.connect(library, arcade, 7)

print(city.describe())

  Arcade -> Bakery(3), Cafe(10), Library(20)
  Bakery -> Cafe(2), Library(9)
    Cafe -> Diner(1)
   Diner -> Bakery(1), Library(2)
 Library -> Arcade(7)


## 4. Run Floyd-Warshall

The runner returns a `FloydWarshallResult` with three things:

- `places`: row and column order for the matrix
- `distances`: final cheapest cost matrix
- `next_hop`: breadcrumbs for rebuilding routes
- `steps`: snapshots for replaying matrix updates

<details>
<summary>Quick check</summary>

The direct `Arcade -> Library` cost is `20`, but the final route should be cheaper through other places.

</details>

In [3]:
runner = FloydWarshallRunner(city)
result = runner.run()
printer = MatrixPrinter(result.places)

print("Final all-pairs cost matrix:")
print(printer.matrix(result.distances))

route = runner.path_between(result, arcade, library)
arcade_index = result.places.index(arcade)
library_index = result.places.index(library)
print("\nBest Arcade -> Library cost:", result.distances[arcade_index][library_index])
print("Best Arcade -> Library route:", printer.path(route))

Final all-pairs cost matrix:
           Arcade  Bakery    Cafe   Diner Library
   Arcade       0       3       5       6       8
   Bakery      12       0       2       3       5
     Cafe      10       2       0       1       3
    Diner       9       1       3       0       2
  Library       7      10      12      13       0

Best Arcade -> Library cost: 8
Best Arcade -> Library route: Arcade -> Bakery -> Cafe -> Diner -> Library


## 5. Replay the Matrix Updates

The replay shows how the matrix changes when each place becomes an allowed middle stop.

<details>
<summary>Hint: dynamic programming angle</summary>

After considering a via place, the matrix answers a slightly bigger question: shortest paths that may use that via place and all earlier via places.

</details>

In [4]:
class MatrixReplay:
    def __init__(self, result: FloydWarshallResult):
        self.result = result
        self.printer = MatrixPrinter(result.places)

    def show(self) -> None:
        for number, step in enumerate(self.result.steps, start=1):
            via = step.via or "initial"
            print(f"Step {number}: via {via}")
            print(" ", step.note)
            print(" ", self.printer.changes(step.changes))
            print(self.printer.matrix(step.distances))
            print()


replay = MatrixReplay(result)
replay.show()

Step 1: via initial
  Start with direct route costs.
  no matrix cells improved
           Arcade  Bakery    Cafe   Diner Library
   Arcade       0       3      10     inf      20
   Bakery     inf       0       2     inf       9
     Cafe     inf     inf       0       1     inf
    Diner     inf       1     inf       0       2
  Library       7     inf     inf     inf       0

Step 2: via Arcade
  Use Arcade as an allowed middle stop.
  Library->Bakery: inf to 10; Library->Cafe: inf to 17
           Arcade  Bakery    Cafe   Diner Library
   Arcade       0       3      10     inf      20
   Bakery     inf       0       2     inf       9
     Cafe     inf     inf       0       1     inf
    Diner     inf       1     inf       0       2
  Library       7      10      17     inf       0

Step 3: via Bakery
  Use Bakery as an allowed middle stop.
  Arcade->Cafe: 10 to 5; Arcade->Library: 20 to 12; Diner->Cafe: inf to 3; Library->Cafe: 17 to 12
           Arcade  Bakery    Cafe   Diner Libr

## 6. Your Experiments

Try changing one thing at a time:

- Add a shortcut from `Cafe` to `Library`
- Remove an expensive direct route
- Make a route two-way
- Add a new place and connect it to the map

<details>
<summary>Challenge</summary>

Predict which matrix cells will improve before you run the replay. Then compare your guess with the output.

</details>

In [5]:
museum = Place("Museum")

experiment = RouteMap()
experiment.connect(arcade, bakery, 3)
experiment.connect(arcade, cafe, 10)
experiment.connect(bakery, cafe, 2)
experiment.connect(cafe, diner, 1)
experiment.connect(cafe, library, 4)
experiment.connect(diner, bakery, 1)
experiment.connect(diner, library, 2)
experiment.connect(library, arcade, 7)
experiment.connect(library, museum, 3)
experiment.connect(museum, cafe, 2)

experiment_runner = FloydWarshallRunner(experiment)
experiment_result = experiment_runner.run()
experiment_printer = MatrixPrinter(experiment_result.places)

print("Experiment matrix:")
print(experiment_printer.matrix(experiment_result.distances))

route = experiment_runner.path_between(experiment_result, arcade, museum)
arcade_index = experiment_result.places.index(arcade)
museum_index = experiment_result.places.index(museum)
print("\nBest Arcade -> Museum cost:", experiment_result.distances[arcade_index][museum_index])
print("Best Arcade -> Museum route:", experiment_printer.path(route))

Experiment matrix:
           Arcade  Bakery    Cafe   Diner Library  Museum
   Arcade       0       3       5       6       8      11
   Bakery      12       0       2       3       5       8
     Cafe      10       2       0       1       3       6
    Diner       9       1       3       0       2       5
  Library       7       7       5       6       0       3
   Museum      12       4       2       3       5       0

Best Arcade -> Museum cost: 11
Best Arcade -> Museum route: Arcade -> Bakery -> Cafe -> Diner -> Library -> Museum


## Visual Trace + Rigor Studio

**Problem frame.** Find every shortest path between every pair of nodes.

**Interactive animation target.** Animate the distance matrix as each intermediate node becomes allowed.

**Correctness handle.** After phase k, each entry is optimal using only the first k nodes as intermediates.

**Complexity handle.** O(V^3) time and O(V^2) space.

**Failure mode to test.** A negative cycle makes shortest distance undefined because cost can decrease forever.

**Studio task.** Choose one matrix cell and narrate the exact intermediate node that improves it.


In [ ]:
from pathlib import Path
import sys

for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "courseware").exists():
        sys.path.insert(0, str(candidate))
        break

from courseware import AlgorithmPlayer, AlgorithmTrace, TraceStep, render_trace_table

# Convert the implementation above into snapshots:
# trace = AlgorithmTrace("Topic trace")
# trace.append("start", {"your_state": ...}, "What changed?", invariant="What remains true?")
# AlgorithmPlayer(trace, your_renderer).display()
print("Use AlgorithmTrace to turn this notebook's algorithm into a step-by-step visual player.")
